In [ ]:
%load_ext autoreload
%autoreload 2

from pathlib import Path
import os
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

PROJECT_ROOT = Path.cwd()
while not (PROJECT_ROOT / "flowmap_legacy").exists():
    if PROJECT_ROOT.parent == PROJECT_ROOT:
        raise RuntimeError("Could not find repository root containing flowmap_legacy")
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from flowmap_legacy.evaluation import evaluate_embedding_method


def load_vector_field(csv_path):
    df = pd.read_csv(csv_path)
    X = df[["x", "y"]].values
    V = df[["vx", "vy"]].values
    time = df["time"].values
    return X, V, time


path_map = {
    "straight_line": "./data/1d/straight_line.csv",
    "sine_curve": "./data/1d/sine_curve.csv",
    "branch_2": "./data/1d/branch_2.csv",
    "branch_4": "./data/1d/branch_4.csv",
    "rotation": "./data/2d/rotation.csv",
    "spiral": "./data/2d/spiral.csv",
    "saddle": "./data/2d/saddle.csv",
    "quadratic_source_sink": "./data/2d/quadratic_source_sink.csv",
}


In [ ]:
import scvelo as scv
import scanpy as sc
import anndata

np.random.seed(42)

# ------------------------------------------
# simulate noisy data (your block unchanged)
# ------------------------------------------
simulation_results = {}
noise, extra_dim = 0.3, 5
for name, path in path_map.items():
    X_gt, V_gt, time = load_vector_field(path)
    X_noisy = X_gt + np.random.normal(scale=noise, size=X_gt.shape)
    V_noisy = V_gt + np.random.normal(scale=noise, size=V_gt.shape)
    X_dummy = np.random.normal(scale=noise, size=(X_gt.shape[0], extra_dim))
    V_dummy = np.random.normal(scale=noise, size=(V_gt.shape[0], extra_dim))
    X = np.hstack([X_noisy, X_dummy])
    V = np.hstack([V_noisy, V_dummy])
    simulation_results[name] = dict(X=X, V=V, X_gt=X_gt, V_gt=V_gt, true_time=time)

In [ ]:
# ------------------------------------------
# PLOT + SCORE IN ONE LOOP
# ------------------------------------------
fig, axs = plt.subplots(1, 8, figsize=(32, 4), constrained_layout=True)
axs = axs.flatten()

records = []   # will collect metric rows

for ax, (name, result) in zip(axs, simulation_results.items()):
    X, V, time  = result["X"], result["V"], result["true_time"]
    X_gt, V_gt  = result["X_gt"], result["V_gt"]

    # ---- scVelo embedding -------------------------------------------------
    adata = anndata.AnnData(X)
    adata.layers["position"] = X
    adata.layers["velocity"] = V
    adata.obs["time"] = (time - time.min()) / (time.max() - time.min())

    scv.pp.neighbors(adata, n_neighbors=30, use_rep="X")
    sc.tl.umap(adata, min_dist=0.3)
    scv.tl.velocity_graph(adata, xkey="position", vkey="velocity")
    scv.tl.velocity_embedding(adata, basis="umap")

    is_2d = name in {
        "rotation",
        "spiral",
        "saddle",
        "quadratic_source_sink"
    }
    
    if is_2d:
        scatter_size = 1200   # larger dots
        scatter_alpha = 0.4  # higher alpha
    else:
        scatter_size = 400
        scatter_alpha = 0.2

    # ---- plotting ---------------------------------------------------------
    scv.pl.velocity_embedding_stream(
        adata,
        basis="umap",
        color="time",
        cmap="viridis",
        ax=ax,
        show=False,
        legend_loc=None,
        colorbar=False,
        density=0.5,
        arrow_size=2.5,
        linewidth=3.0,
        alpha=scatter_alpha,
        size=scatter_size,
    )

    ax.set_aspect("equal")

    # ---- scoring ----------------------------------------------------------
    X_emb = adata.obsm["X_umap"]
    V_emb = adata.obsm["velocity_umap"]       # raw projected velocities
    scores = evaluate_embedding_method(X_gt, X_emb, V_gt, V_emb, k=30)
    scores["dataset"] = name
    records.append(scores)

fig_dir = "./figures/simulation"
os.makedirs(fig_dir, exist_ok=True)

fig_path = os.path.join(fig_dir, "scvelo_embedding_streams.png")
plt.savefig(fig_path, dpi=300, bbox_inches="tight")
print(f"Saved figure to {fig_path}")
plt.show()

# ------------------------------------------
# RESULTS TABLE → CSV
# ------------------------------------------
scores_df = pd.DataFrame(records).set_index("dataset")
os.makedirs("./data/8_vf_collection", exist_ok=True)
scores_df.to_csv("./data/8_vf_collection/scvelo.csv")
print("\nSaved score table to ./data/8_vf_collection/scvelo.csv\n")
print(scores_df.round(4))